In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from datetime import timedelta

# Load data
df_flow = pd.read_parquet("filterflow_data.parquet")
df_error = pd.read_parquet("error_data.parquet")

# Determine the correct column name for filter flow
if 'filterflow' in df_flow.columns:
    filter_flow_column = 'filterflow'
elif 'filter_flow' in df_flow.columns:
    filter_flow_column = 'filter_flow'
else:
    print("Available columns:", df_flow.columns.tolist())
    raise ValueError("Could not find filter flow column in the data")

# Convert timestamp to datetime for both datasets
df_flow['timestamp'] = pd.to_datetime(df_flow['timestamp'], format='%d.%m.%Y %H:%M')
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], format='%d.%m.%Y %H:%M')

# Filter to include only printers 1-10
printers_to_include = list(range(1, 31))
df_flow = df_flow[df_flow['printer'].isin(printers_to_include)]
df_error = df_error[df_error['printer'].isin(printers_to_include)]

# Error heatmap by printer and color
error_counts = df_error.groupby(['printer', 'color']).size().reset_index(name='error_count')
plt.figure(figsize=(12, 8))
pivot_errors = pd.pivot_table(error_counts, values='error_count', index='printer', columns='color', fill_value=0)
sns.heatmap(pivot_errors, annot=True, cmap='Reds', fmt='g')
plt.title('Error Distribution by Printer and Color')
plt.tight_layout()
plt.show()

# Error timeline - when do errors occur?
df_error['date'] = df_error['timestamp'].dt.date
errors_by_date = df_error.groupby('date').size().reset_index(name='error_count')
plt.figure(figsize=(14, 6))
plt.plot(errors_by_date['date'], errors_by_date['error_count'], marker='o', linestyle='-')
plt.xlabel('Date')
plt.ylabel('Number of Errors')
plt.title('Error Frequency Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate average filter flow for each printer and color
avg_flow = df_flow.groupby(['printer', 'color'])[filter_flow_column].mean().reset_index()

# Merge with error counts
flow_vs_errors = pd.merge(avg_flow, error_counts, on=['printer', 'color'], how='outer').fillna(0)

# Create scatter plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    flow_vs_errors[filter_flow_column], 
    flow_vs_errors['error_count'],
    c=flow_vs_errors['error_count'],
    cmap='viridis',
    s=100,
    alpha=0.7
)

# Add labels for each point
for i, row in flow_vs_errors.iterrows():
    plt.annotate(
        f"P{int(row['printer'])}-{row['color']}", 
        (row[filter_flow_column], row['error_count']),
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.colorbar(scatter, label='Error Count')
plt.xlabel('Average Filter Flow')
plt.ylabel('Number of Errors')
plt.title('Relationship Between Average Filter Flow and Error Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate filter flow variability (standard deviation)
flow_std = df_flow.groupby(['printer', 'color'])[filter_flow_column].std().reset_index()
flow_std = flow_std.rename(columns={filter_flow_column: 'flow_std'})

# Merge with error counts
std_vs_errors = pd.merge(flow_std, error_counts, on=['printer', 'color'], how='outer').fillna(0)

# Create scatter plot
plt.figure(figsize=(12, 8))
plt.scatter(
    std_vs_errors['flow_std'], 
    std_vs_errors['error_count'],
    c=std_vs_errors['error_count'],
    cmap='viridis',
    s=100,
    alpha=0.7
)

# Add labels for each point
for i, row in std_vs_errors.iterrows():
    plt.annotate(
        f"P{int(row['printer'])}-{row['color']}", 
        (row['flow_std'], row['error_count']),
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.xlabel('Filter Flow Standard Deviation')
plt.ylabel('Number of Errors')
plt.title('Relationship Between Filter Flow Variability and Error Frequency')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Function to get filter flow before each error
def get_flow_before_error(error_row, window_hours=24):
    printer = error_row['printer']
    color = error_row['color']
    error_time = error_row['timestamp']
    
    # Find filter flow values before this error
    before_error = df_flow[
        (df_flow['printer'] == printer) &
        (df_flow['color'] == color) &
        (df_flow['timestamp'] < error_time) &
        (df_flow['timestamp'] >= error_time - timedelta(hours=window_hours))
    ]
    
    if len(before_error) > 0:
        return before_error[filter_flow_column].mean()
    return np.nan

# Apply to each error
df_error['flow_before_error'] = df_error.apply(get_flow_before_error, axis=1)
df_error_with_flow = df_error.dropna(subset=['flow_before_error'])

# Prepare comparison data
comparison_data = []
for (printer, color), group in df_error_with_flow.groupby(['printer', 'color']):
    # Average flow before errors
    pre_error_flow = group['flow_before_error'].mean()
    
    # Average flow during normal operation
    all_flow = df_flow[(df_flow['printer'] == printer) & 
                      (df_flow['color'] == color)][filter_flow_column].mean()
    
    comparison_data.append({
        'printer': printer,
        'color': color,
        'normal_flow': all_flow,
        'pre_error_flow': pre_error_flow,
        'difference': pre_error_flow - all_flow,
        'error_count': len(group)
    })

comparison_df = pd.DataFrame(comparison_data)

# Sort by error count and show top entries
comparison_df = comparison_df.sort_values('error_count', ascending=False)
top_n = min(10, len(comparison_df))

# Plot comparison
plt.figure(figsize=(14, 8))
index = np.arange(top_n)
bar_width = 0.35

plt.bar(index, comparison_df.head(top_n)['normal_flow'], bar_width, 
       label='Normal Operation', color='green', alpha=0.7)
plt.bar(index + bar_width, comparison_df.head(top_n)['pre_error_flow'], bar_width,
       label='Before Error', color='red', alpha=0.7)

plt.xlabel('Printer-Color Combination')
plt.ylabel('Average Filter Flow')
plt.title('Filter Flow: Normal Operation vs. Before Errors')
plt.xticks(index + bar_width/2, 
          [f"P{int(row['printer'])}-{row['color']}" for _, row in comparison_df.head(top_n).iterrows()],
          rotation=45)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Select a printer-color combination with high error count
if not error_counts.empty:
    top_error = error_counts.sort_values('error_count', ascending=False).iloc[0]
    focus_printer = top_error['printer']
    focus_color = top_error['color']
    
    # Get flow data and errors for this printer-color
    flow_data = df_flow[
        (df_flow['printer'] == focus_printer) &
        (df_flow['color'] == focus_color)
    ].sort_values('timestamp')
    
    error_data = df_error[
        (df_error['printer'] == focus_printer) &
        (df_error['color'] == focus_color)
    ]['timestamp']
    
    # Create time series plot
    plt.figure(figsize=(15, 6))
    plt.plot(flow_data['timestamp'], flow_data[filter_flow_column], 'b-', alpha=0.7, label='Filter Flow')
    
    # Add vertical lines for errors
    y_min, y_max = plt.ylim()
    for error_time in error_data:
        plt.axvline(x=error_time, color='red', linestyle='--', alpha=0.7)
    
    # Add a small sample of error markers in the legend
    plt.plot([], [], 'r--', label='Error Occurrence')
    
    plt.xlabel('Time')
    plt.ylabel('Filter Flow')
    plt.title(f'Filter Flow Time Series with Error Markers - Printer {focus_printer}, Color {focus_color}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Create bins for filter flow values
df_flow['flow_bin'] = pd.cut(df_flow[filter_flow_column], bins=8)

# Function to check if an error followed within 24 hours
def had_subsequent_error(flow_row):
    printer = flow_row['printer']
    color = flow_row['color']
    flow_time = flow_row['timestamp']
    
    # Check if an error occurred within the next 24 hours
    errors_after = df_error[
        (df_error['printer'] == printer) &
        (df_error['color'] == color) &
        (df_error['timestamp'] > flow_time) &
        (df_error['timestamp'] <= flow_time + timedelta(hours=24))
    ]
    
    return 1 if len(errors_after) > 0 else 0

# Apply to a sample (this can be slow on large datasets)
sample_size = min(5000, len(df_flow))
flow_sample = df_flow.sample(sample_size, random_state=42)
flow_sample['had_error'] = flow_sample.apply(had_subsequent_error, axis=1)

# Calculate error probability by flow bin
error_prob = flow_sample.groupby('flow_bin')['had_error'].mean().reset_index()
error_prob = error_prob.sort_values('flow_bin')

# Plot
plt.figure(figsize=(12, 6))
bars = plt.bar(
    error_prob['flow_bin'].astype(str),
    error_prob['had_error'] * 100,  # Convert to percentage
    color='orange',
    alpha=0.7
)

plt.xlabel('Filter Flow Range')
plt.ylabel('Error Probability (%)')
plt.title('Probability of Error Following Different Filter Flow Ranges')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2.,
        height + 0.5,
        f'{height:.1f}%',
        ha='center', 
        va='bottom'
    )

plt.tight_layout()
plt.show()

In [ ]:
# For each color, compare filter flow patterns before errors
colors = df_flow['color'].unique()

# Calculate grid size based on number of colors
n_colors = len(colors)
n_cols = 2  # You can adjust this if you want more or fewer columns
n_rows = (n_colors + n_cols - 1) // n_cols  # Ceiling division to get enough rows

plt.figure(figsize=(15, 5 * n_rows))  # Adjust height based on number of rows
for i, color in enumerate(colors):
    plt.subplot(n_rows, n_cols, i+1)
    
    # Get errors for this color
    color_errors = df_error[df_error['color'] == color]
    
    # Get average filter flow before errors by printer
    printer_avg = []
    printer_ids = []
    
    for printer in printers_to_include:
        printer_errors = color_errors[color_errors['printer'] == printer]
        if len(printer_errors) > 0:
            avg_before = 0
            count = 0
            
            for _, error in printer_errors.iterrows():
                flow_before = df_flow[
                    (df_flow['printer'] == printer) &
                    (df_flow['color'] == color) &
                    (df_flow['timestamp'] < error['timestamp']) &
                    (df_flow['timestamp'] >= error['timestamp'] - timedelta(hours=24))
                ][filter_flow_column].mean()
                
                if not np.isnan(flow_before):
                    avg_before += flow_before
                    count += 1
            
            if count > 0:
                printer_avg.append(avg_before / count)
                printer_ids.append(printer)
    
    # Plot if we have data
    if printer_ids:
        plt.bar(printer_ids, printer_avg, alpha=0.7, 
               color=color.lower() if color.lower() in ['c', 'm', 'y', 'k'] else 'blue')
        plt.axhline(y=df_flow[df_flow['color'] == color][filter_flow_column].mean(), 
                   color='red', linestyle='--', label='Average')
        plt.xlabel('Printer')
        plt.ylabel('Avg Filter Flow Before Errors')
        plt.title(f'Color {color} - Filter Flow Before Errors')
        plt.legend()
    else:
        plt.text(0.5, 0.5, f'No errors for Color {color}', 
                horizontalalignment='center', verticalalignment='center')
    
plt.tight_layout()
plt.show()

In [ ]:
# Calculate z-scores for each printer-color combination
df_flow['z_score'] = 0.0

for printer in df_flow['printer'].unique():
    for color in df_flow['color'].unique():
        mask = (df_flow['printer'] == printer) & (df_flow['color'] == color)
        if df_flow[mask].shape[0] > 1:  # Need at least 2 points to calculate z-score
            mean = df_flow.loc[mask, filter_flow_column].mean()
            std = df_flow.loc[mask, filter_flow_column].std()
            if std > 0:  # Avoid division by zero
                df_flow.loc[mask, 'z_score'] = (df_flow.loc[mask, filter_flow_column] - mean) / std

# Function to check if an error followed an anomalous reading
def error_after_anomaly(flow_row, z_threshold=2.0, hours=24):
    if abs(flow_row['z_score']) < z_threshold:
        return 0  # Not an anomaly
    
    printer = flow_row['printer']
    color = flow_row['color']
    flow_time = flow_row['timestamp']
    
    # Check if an error followed this anomaly
    errors_after = df_error[
        (df_error['printer'] == printer) &
        (df_error['color'] == color) &
        (df_error['timestamp'] > flow_time) &
        (df_error['timestamp'] <= flow_time + timedelta(hours=hours))
    ]
    
    return 1 if len(errors_after) > 0 else 0

# Apply to a sample
sample = df_flow.sample(min(5000, len(df_flow)), random_state=42)
sample['error_followed'] = sample.apply(error_after_anomaly, axis=1)

# Calculate error rates for normal vs anomalous readings
anomalies = sample[abs(sample['z_score']) >= 2.0]
normal = sample[abs(sample['z_score']) < 2.0]

anomaly_error_rate = anomalies['error_followed'].mean() * 100
normal_error_rate = normal['error_followed'].mean() * 100

# Create comparison bar chart
plt.figure(figsize=(10, 6))
plt.bar(['Normal Readings', 'Anomalous Readings'], 
       [normal_error_rate, anomaly_error_rate],
       color=['green', 'red'], alpha=0.7)
plt.ylabel('Error Probability (%)')
plt.title('Error Probability: Normal vs Anomalous Filter Flow Readings')
plt.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate([normal_error_rate, anomaly_error_rate]):
    plt.text(i, v + 0.5, f'{v:.1f}%', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import calendar

# Add day of week and hour columns
df_error['day_of_week'] = df_error['timestamp'].dt.day_name()
df_error['hour'] = df_error['timestamp'].dt.hour

# Create day/hour heatmap
plt.figure(figsize=(12, 8))
days_order = list(calendar.day_name)  # To ensure proper day order
error_heat = pd.crosstab(df_error['day_of_week'], df_error['hour'])
error_heat = error_heat.reindex(days_order)

# Changed fmt from 'd' to '.0f' to handle float values
sns.heatmap(error_heat, cmap='YlOrRd', annot=True, fmt='.0f', cbar_kws={'label': 'Error Count'})
plt.title('Error Frequency by Day of Week and Hour (Printers 1-10)')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate filter flow stability metrics
window_size = 3  # Number of readings to use for stability calculation

# Initialize a dictionary to store results
stability_data = []

# Process each printer-color combination
for printer in printers_to_include:
    for color in df_flow['color'].unique():
        # Get data for this printer and color
        subset = df_flow[(df_flow['printer'] == printer) & 
                         (df_flow['color'] == color)].sort_values('timestamp')
        
        if len(subset) > window_size:
            # Calculate rolling metrics
            subset['rolling_std'] = subset[filter_flow_column].rolling(window=window_size).std()
            
            # For each error, find the average stability before it
            printer_errors = df_error[(df_error['printer'] == printer) & 
                                     (df_error['color'] == color)]
            
            for _, error in printer_errors.iterrows():
                # Find readings before this error
                before_error = subset[
                    (subset['timestamp'] < error['timestamp']) & 
                    (subset['timestamp'] >= error['timestamp'] - timedelta(hours=48))
                ]
                
                if not before_error.empty and not before_error['rolling_std'].isna().all():
                    stability_data.append({
                        'printer': printer,
                        'color': color,
                        'pre_error_stability': before_error['rolling_std'].mean()
                    })

# Convert to DataFrame
stability_df = pd.DataFrame(stability_data)

if not stability_df.empty:
    # Group by printer and color
    avg_stability = stability_df.groupby(['printer', 'color'])['pre_error_stability'].mean().reset_index()
    
    # Pivot for visualization
    pivot_stability = avg_stability.pivot(index='printer', columns='color', values='pre_error_stability')
    
    # Create heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_stability, annot=True, cmap='viridis_r', fmt='.3f')
    plt.title('Filter Flow Stability Before Errors (Lower = More Stable)')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to calculate stability metrics")

In [ ]:
# Function to get filter flow patterns after errors
def analyze_error_recovery(window_before=24, window_after=72):
    recovery_data = []
    
    for _, error in df_error.iterrows():
        printer = error['printer']
        color = error['color']
        error_time = error['timestamp']
        
        # Get readings before error
        before = df_flow[
            (df_flow['printer'] == printer) &
            (df_flow['color'] == color) &
            (df_flow['timestamp'] < error_time) &
            (df_flow['timestamp'] >= error_time - timedelta(hours=window_before))
        ]
        
        # Get readings after error
        after = df_flow[
            (df_flow['printer'] == printer) &
            (df_flow['color'] == color) &
            (df_flow['timestamp'] > error_time) &
            (df_flow['timestamp'] <= error_time + timedelta(hours=window_after))
        ]
        
        if not before.empty and not after.empty:
            # Calculate averages
            before_avg = before[filter_flow_column].mean()
            after_avg = after[filter_flow_column].mean()
            
            recovery_data.append({
                'printer': printer,
                'color': color,
                'error_time': error_time,
                'before_avg': before_avg,
                'after_avg': after_avg,
                'percent_change': ((after_avg - before_avg) / before_avg) * 100 if before_avg != 0 else np.nan
            })
    
    return pd.DataFrame(recovery_data)

recovery_df = analyze_error_recovery()

if not recovery_df.empty:
    # Group by printer and color
    avg_recovery = recovery_df.groupby(['printer', 'color'])['percent_change'].mean().reset_index()
    
    # Prepare for visualization
    pivot_recovery = avg_recovery.pivot(index='printer', columns='color', values='percent_change')
    
    # Create heatmap of recovery percentages
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_recovery, annot=True, cmap='RdYlGn', fmt='.1f')
    plt.title('Average % Change in Filter Flow After Errors')
    plt.tight_layout()
    plt.show()
    
    # Show distribution of recovery patterns
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='color', y='percent_change', data=recovery_df)
    plt.axhline(y=0, color='red', linestyle='--')
    plt.title('Distribution of Filter Flow Changes After Errors by Color')
    plt.ylabel('% Change After Error')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data to analyze error recovery patterns")

In [ ]:
# Create histograms of filter flow with error markers
def plot_flow_distributions():
    colors = df_flow['color'].unique()
    n_colors = len(colors)
    n_cols = 2
    n_rows = (n_colors + n_cols - 1) // n_cols
    
    plt.figure(figsize=(15, 5 * n_rows))
    
    for i, color in enumerate(colors):
        plt.subplot(n_rows, n_cols, i+1)
        
        # Get all filter flow values for this color
        all_flows = df_flow[df_flow['color'] == color][filter_flow_column]
        
        # Get filter flow values associated with errors
        error_flows = []
        for _, error in df_error[df_error['color'] == color].iterrows():
            printer = error['printer']
            error_time = error['timestamp']
            
            # Get the closest reading before the error
            before = df_flow[
                (df_flow['printer'] == printer) &
                (df_flow['color'] == color) &
                (df_flow['timestamp'] < error_time)
            ].sort_values('timestamp', ascending=False).head(1)
            
            if not before.empty:
                error_flows.append(before[filter_flow_column].iloc[0])
        
        # Plot the histogram
        if not all_flows.empty:
            plt.hist(all_flows, bins=20, alpha=0.5, color='blue', label='All Readings')
            
            if error_flows:
                plt.hist(error_flows, bins=10, alpha=0.7, color='red', label='Before Errors')
                
                # Add vertical lines at mean and median
                plt.axvline(np.mean(error_flows), color='red', linestyle='--', 
                            label=f'Error Mean: {np.mean(error_flows):.2f}')
                plt.axvline(np.median(error_flows), color='darkred', linestyle=':',
                            label=f'Error Median: {np.median(error_flows):.2f}')
            
            plt.axvline(all_flows.mean(), color='blue', linestyle='--',
                        label=f'Overall Mean: {all_flows.mean():.2f}')
            
            plt.title(f'Filter Flow Distribution - Color {color}')
            plt.xlabel('Filter Flow')
            plt.ylabel('Frequency')
            plt.legend()
        else:
            plt.text(0.5, 0.5, f'No data for Color {color}',
                    horizontalalignment='center', verticalalignment='center')
    
    plt.tight_layout()
    plt.show()

plot_flow_distributions()

In [ ]:
# Calculate time since last error for each printer-color
error_intervals = []

for printer in printers_to_include:
    for color in df_flow['color'].unique():
        # Get errors for this printer-color
        errors = df_error[(df_error['printer'] == printer) & 
                         (df_error['color'] == color)].sort_values('timestamp')
        
        if len(errors) >= 2:
            # Calculate time differences
            errors['hours_since_last'] = errors['timestamp'].diff().dt.total_seconds() / 3600
            
            for _, error in errors.iloc[1:].iterrows():
                error_intervals.append({
                    'printer': printer,
                    'color': color,
                    'hours_since_last': error['hours_since_last']
                })

intervals_df = pd.DataFrame(error_intervals)

if not intervals_df.empty:
    # Create histogram of intervals
    plt.figure(figsize=(12, 6))
    sns.histplot(intervals_df['hours_since_last'], bins=30, kde=True)
    plt.xlabel('Hours Since Previous Error')
    plt.ylabel('Frequency')
    plt.title('Distribution of Time Intervals Between Errors')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Show intervals by printer and color
    plt.figure(figsize=(14, 8))
    sns.boxplot(x='printer', y='hours_since_last', hue='color', data=intervals_df)
    plt.yscale('log')  # Log scale often works better for time intervals
    plt.xlabel('Printer')
    plt.ylabel('Hours Since Last Error (log scale)')
    plt.title('Error Clustering Patterns by Printer and Color')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough sequential errors to analyze intervals")

In [ ]:
# Count errors by date for each printer
df_error['date'] = df_error['timestamp'].dt.date
error_counts_by_printer_date = df_error.groupby(['date', 'printer']).size().unstack(fill_value=0)

# Calculate correlation between printer errors
error_correlation = error_counts_by_printer_date.corr()

# Create heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(error_correlation, dtype=bool))  # Create mask for upper triangle
sns.heatmap(error_correlation, annot=True, cmap='coolwarm', center=0, mask=mask, 
           vmin=-1, vmax=1, fmt='.2f')
plt.title('Correlation of Error Occurrence Between Printers')
plt.tight_layout()
plt.show()